In [1]:
import cvxpy as cp
import numpy as np

In [2]:
mu = np.array([0.3, 0.5, 0.2])
sigma = np.array([
    [0.10, 0.02, 0.04],
    [0.02, 0.08, 0.01],
    [0.04, 0.01, 0.12]
])
gamma = 2

Variant 1: Unconstrained mean variance optimization has a closed-form solution which is the direct inverse of the covariance matrix scaled by returns where $\gamma$ is the risk-aversion penalty:
$$\max_w \ \mu^Tw - \frac{\gamma}{2}w^T\Sigma w$$
Take the gradient with respoect to $w$ and set to zero:
$$\mu-\gamma \Sigma w = 0$$
Solve for $w$:
$$\boxed{w = \frac{1}{\gamma} \Sigma^{-1} \mu}$$

In [3]:
def unconstrained_mean_variance(mu, sigma, gamma):
    Sinv_mu   = np.linalg.solve(sigma, mu)      # Sigma^-1 mu
    w = Sinv_mu / gamma
    print(f"Target expected Returns: {mu @ w.T:.2%}")
    print(f"Target variance: {w @ sigma @ w.T:.2%}")
    print(f"Objective function value: {mu @ w.T - gamma * w @ sigma @ w.T / 2:.4f}")
    print(f"Optimal weights: {w.round(4)}")
    print(f"Gradient value: {(mu - gamma * sigma @ w).round(4)}")
unconstrained_mean_variance(mu, sigma, gamma)

Target expected Returns: 174.62%
Target variance: 87.31%
Objective function value: 0.8731
Optimal weights: [0.7911 2.8861 0.3291]
Gradient value: [ 0.  0. -0.]


Variant 2: Mean variance optimization with equailty constraint where weights sum to exactly 1.
$$\max_w \ \mu^Tw - \frac{\gamma}{2}w^T\Sigma w \quad \text{s.t.} \quad \mathbf{1}^Tw = 1$$
Set up the Lagrangian with the budget constraint:
$$\mathcal{L} = \mu^Tw - \frac{\gamma}{2}w^T\Sigma w - \lambda(\mathbf{1}^Tw - 1)$$
Take the gradient with respect to $w$ and set to zero:
$$\nabla_w \mathcal{L} = \mu - \gamma\Sigma w - \lambda\mathbf{1} = 0$$
Solve for $w$:
$$w = \frac{1}{\gamma}\Sigma^{-1}(\mu - \lambda\mathbf{1})$$
Now pin down $\lambda$ using the budget constraint $\mathbf{1}^Tw = 1$. Substitute:
$$\mathbf{1}^T \cdot \frac{1}{\gamma}\Sigma^{-1}(\mu - \lambda\mathbf{1}) = 1$$
$$\frac{1}{\gamma}\left(\mathbf{1}^T\Sigma^{-1}\mu - \lambda\,\mathbf{1}^T\Sigma^{-1}\mathbf{1}\right) = 1$$
Let $A = \mathbf{1}^T\Sigma^{-1}\mathbf{1}$ and $B = \mathbf{1}^T\Sigma^{-1}\mu$:
$$\lambda = \frac{B - \gamma}{A}​$$
Substitute back:
$$\boxed{w = \frac{1}{\gamma}\Sigma^{-1}\left(\mu - \frac{B - \gamma}{A}\mathbf{1}\right)}​$$

In [4]:
def mean_variance_with_budget_constraint(mu, sigma, gamma):
    n = len(mu)
    ones = np.ones(n)
    
    # closed form — pure equality-constrained QP
    Sinv_mu   = np.linalg.solve(sigma, mu)      # Sigma^-1 mu
    Sinv_one  = np.linalg.solve(sigma, ones)    # Sigma^-1 1
    A = ones @ Sinv_one
    B = ones @ Sinv_mu
    lam = (B - gamma) / A
    w = (Sinv_mu - lam * Sinv_one) / gamma
    print(f"Target expected Returns: {mu @ w.T:.2%}")
    print(f"Target variance: {w @ sigma @ w.T:.2%}")
    print(f"Objective function value: {mu @ w.T - gamma * w @ sigma @ w.T / 2:.4f}")
    print(f"Optimal weights: {w.round(4)}")
    print(f"Constraint value: {sum(w):.2%}")
    print(f"Shadow price: {lam:.4f}")
mean_variance_with_budget_constraint(mu, sigma, gamma)

Target expected Returns: 63.33%
Target variance: 17.78%
Objective function value: 0.4556
Optimal weights: [ 0.      1.4444 -0.4444]
Constraint value: 100.00%
Shadow price: 0.2778


Variant 3: Mean variance optimization with budget constraint and bounds.

In [5]:
def long_only_mean_variance(mu, sigma, gamma):
    n = len(mu)
    # with constraints — fall back to CVXPY
    w = cp.Variable(n)
    obj = cp.Maximize(mu @ w - (gamma/2) * cp.quad_form(w, cp.psd_wrap(sigma)))
    cons = [cp.sum(w) == 1, w >= 0]
    prob = cp.Problem(obj, cons)
    prob.solve(solver=cp.OSQP)
    if prob.status not in ("optimal", "optimal_inaccurate"):
        raise ValueError(f"solve failed: {prob.status}")
    print(f"Target expected Returns: {mu @ w.value.T:.2%}")
    print(f"Target variance: {w.value @ sigma @ w.value.T:.2%}")
    print(f"Objective function value: {mu @ w.value.T - gamma * w.value @ sigma @ w.value.T / 2:.4f}")
    print(f"Optimal weights: {w.value.round(4)}")
    print(f"Constraint value: {sum(w.value):.2%}")
long_only_mean_variance(mu, sigma, gamma)

Target expected Returns: 50.00%
Target variance: 8.00%
Objective function value: 0.4200
Optimal weights: [0. 1. 0.]
Constraint value: 100.00%
